# Feature 013 — robust FCR bidding in action

This source-only walkthrough demonstrates a genuine joint choice rather than a flat-price smoke test. In one 24-hour problem, a 20 kWh battery first uses its power headroom and energy headroom for day-ahead arbitrage, then offers only the symmetric FCR capacity that remains robustly supportable.

The notebook answers four concrete questions:

1. Which hourly energy baseline and four-hour reserve blocks does the bidder select?
2. Why is selected reserve below the 10 kW converter rating?
3. Which power or stored-energy limits bind in each block?
4. What happens when activation that was hidden from bid selection is supplied later to Feature 012?


## Setup, terminology, and evidence limits

Run from the repository root after:

    uv sync --group notebook
    uv run jupyter lab

The source notebook is committed with empty outputs. Default execution is deterministic and offline: HiGHS solves one authored problem, post-acceptance activation is authored locally, and the Julia/Gurobi evidence is a small normalized fixture.

The default teaching case is entirely synthetic and purpose-built in [build_walkthrough_problem](./run_small_case.py): the first eight hourly day-ahead prices are 0.03, 0.02, 0.01, 0.04, 0.18, 0.16, 0.14, and 0.12 EUR/kWh, the remaining sixteen are 0.06 EUR/kWh, and every four-hour FCR block is valued at 0.10 EUR/kW/block. These values create an interpretable arbitrage-versus-reserve choice; they are not RTE, ENTSO-E, or other observed market data. The inherited delivery date, 18 June 2023, supplies an ordinary-day time mapping only and does not imply historical price provenance. The normalized historical fixture examined in Section 8 is a separate case.

- **Restriction**: the conservative formulation. It may emit a robust-labelled FixedFCRBidPlan only after successful solution, zero modeled slack within tolerance, and an independent residual audit.
- **Relaxation**: an optimistic upper-bound formulation. Its decisions are diagnostic and are never presented as robustly deployable.
- **Activation**: evaluation-only information. Feature 013 never receives the realized activation samples while selecting the bid.
- **Reference evidence**: three-decimal source decisions define closed precision intervals of half-width 0.0005 kW. Compatibility with those intervals is not proof of identical unrounded decisions, source optimality, or complete Julia trace parity.

The authored case uses unit charge/discharge efficiencies so every binding limit can be checked by hand. Its post-acceptance handoff keeps Feature 013's ex-ante optimization objective separate from Feature 012's ex-post settlement. The governed historical fixture retains the source battery's lossy physics for selected parity work.


In [ ]:
from pathlib import Path

RUN_NATIVE_REFERENCE_BENCHMARK = False
RUN_OPTIONAL_GUROBI_REFERENCE_BENCHMARK = False
RUN_EXTERNAL_ACTIVATION_REPLAY = False
EXPORT_FIGURES = False

REFERENCE_BENCHMARK_TIME_LIMIT_SECONDS = 3600.0
GUROBI_REFERENCE_BENCHMARK_TIME_LIMIT_SECONDS = 3600.0
REFERENCE_FIXTURE_PATH = Path(
    "tests/fixtures/fcr_bidding/"
    "robust_storage_bidding_2023-06-18_restriction_v1.json"
)
EXTERNAL_ACTIVATION_PATH = Path(
    "artifacts/reference/robust_storage_bidding/regulation_signal.csv"
)
EVALUATION_FIXTURE_PATH = Path(
    "tests/fixtures/fcr_bid_evaluation/"
    "robust_storage_bidding_2023-06-18_aggregate_v1.json"
)
FIGURE_EXPORT_DIR = Path("tmp/robust-fcr-bidding-notebook/figures")


In [ ]:
from dataclasses import replace
from datetime import UTC, datetime
import hashlib
import json

import matplotlib.pyplot as plt
import pandas as pd

from examples.robust_fcr_bidding.run_small_case import build_walkthrough_problem
from open_energy_lab.control import (
    RobustFCRSolvePolicy,
    compare_robust_fcr_bounds,
    solve_robust_fcr_bid,
)
from oel_validation.robust_fcr_reference import (
    build_reference_decision_intervals,
    build_robust_fcr_reference_problem,
    compare_robust_fcr_reference,
    load_robust_fcr_reference_fixture,
)
from open_energy_lab.market import (
    ActivationImportRequest,
    CanonicalInterval,
    import_regulation_activation_trace,
)
from open_energy_lab.simulator import (
    BatteryComponent,
    EvaluationMarketContext,
    EvaluationPriceObservation,
    FCRBidEvaluationRequest,
    MultiRateTimeContract,
    evaluate_fixed_fcr_bid,
)

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)
generated_figures = {}

def stable_fingerprint(value):
    payload = json.dumps(
        value,
        allow_nan=False,
        ensure_ascii=False,
        separators=(",", ":"),
        sort_keys=True,
    ).encode("utf-8")
    return hashlib.sha256(payload).hexdigest()

def annotate_bars(axis, bars, *, digits=3):
    for bar in bars:
        height = bar.get_height()
        axis.annotate(
            f"{height:.{digits}f}",
            (bar.get_x() + bar.get_width() / 2.0, height),
            xytext=(0, 4 if height >= 0 else -12),
            textcoords="offset points",
            ha="center",
            va="bottom" if height >= 0 else "top",
            fontsize=8,
        )


## 1. Ownership, signs, and information flow

Feature 013 owns bid selection. Feature 012 alone owns accepted-plan physical replay and settlement. Public battery power is charge-positive, while the pinned source energy variable is discharge-positive:

$$
p_h^{\mathrm{OEL}}=-x_h^{0,\mathrm{source}},
\qquad
r_b^{\mathrm{OEL}}=x_b^{r,\mathrm{source}}\ge 0.
$$

Only after selection and acceptance does realized activation $a_k\in[-1,1]$ enter the Feature 012 request:

$$
p_k^{\mathrm{requested}}
=
p_{h(k)}^{\mathrm{OEL}}-a_k r_{b(k)}.
$$

Positive activation therefore requests upward discharge, which is negative OEL battery power. Each four quarter-hours maps to one hourly energy decision, and each sixteen quarter-hours maps to one four-hour FCR decision.


In [ ]:
walkthrough_problem = build_walkthrough_problem()
time_contract = walkthrough_problem.time_contract
battery = walkthrough_problem.battery
recovery_budget = walkthrough_problem.uncertainty.recovery_window
horizon_budget = walkthrough_problem.uncertainty.horizon_wide

case_table = pd.DataFrame(
    [
        ("planning intervals", len(time_contract.planning_intervals), "count"),
        ("hourly energy decisions", len(time_contract.energy_intervals), "count"),
        ("four-hour FCR decisions", len(time_contract.fcr_intervals), "count"),
        ("charge power limit", battery.maximum_charge_power_kw, "kW"),
        ("discharge power limit", battery.maximum_discharge_power_kw, "kW"),
        ("stored-energy range", f"{battery.minimum_stored_energy_kwh:g} to {battery.maximum_stored_energy_kwh:g}", "kWh"),
        ("initial stored energy", battery.initial_stored_energy_kwh, "kWh"),
        ("terminal minimum", walkthrough_problem.terminal_rule.minimum_terminal_stored_energy_kwh, "kWh"),
        ("charge efficiency", battery.charge_efficiency, "fraction"),
        ("discharge efficiency", battery.discharge_efficiency, "fraction"),
    ],
    columns=["quantity", "value", "unit"],
)
case_table


In [ ]:
mapping_table = pd.DataFrame(
    [
        {
            "planning interval": index,
            "hourly energy index": time_contract.planning_to_energy[index],
            "four-hour FCR index": time_contract.planning_to_fcr[index],
        }
        for index in range(96)
    ]
)
derivation = horizon_budget.derivation_inputs
uncertainty_table = pd.DataFrame(
    [
        ("recovery activation budget gamma-prime", recovery_budget.recovery_activation_budget_hours, "h"),
        ("recovery window Gamma-prime", recovery_budget.recovery_window_hours, "h"),
        ("horizon T", horizon_budget.horizon_hours, "h"),
        ("full recovery windows", derivation["full_window_count"], "count"),
        ("horizon remainder", derivation["remainder_hours"], "h"),
        ("derived horizon budget gamma", horizon_budget.horizon_activation_budget_hours, "h"),
    ],
    columns=["quantity", "value", "unit"],
)
display(mapping_table.head(20), uncertainty_table)


### Two distinct uncertainty sets

The recovery-window set and horizon-wide set are separate immutable contracts. The non-intraday formulation uses the conservative covering budget

$$
n=\left\lfloor\frac{T}{\Gamma'}\right\rfloor,
\qquad
\gamma
=
n\gamma'
+
\min\!\left(T-n\Gamma',\,\gamma'\right).
$$

For $T=24$ h, $\gamma'=0.25$ h, and $\Gamma'=2.25$ h:

$$
n=10,
\qquad
\gamma=10(0.25)+\min(1.5,0.25)=2.75\ \mathrm{h}.
$$

This 2.75 h value is the horizon-wide robust budget used during bid selection. It is not a realized activation forecast.


In [ ]:
hour_indices = list(range(24))
fcr_block_indices = list(range(6))
energy_prices = walkthrough_problem.prices.day_ahead_energy_prices_eur_per_kwh
fcr_prices = walkthrough_problem.prices.fcr_capacity_prices_eur_per_kw_block

price_figure, price_axes = plt.subplots(2, 1, figsize=(10, 6.5), constrained_layout=True)
price_axes[0].step(
    list(range(25)),
    [*energy_prices, energy_prices[-1]],
    where="post",
    color="tab:blue",
    linewidth=2,
)
price_axes[0].set_xlim(0, 24)
price_axes[0].set_ylabel("Day-ahead price (EUR/kWh)")
price_axes[0].set_xlabel("Delivery hour")
price_axes[0].set_title("Synthetic input 1: a price spread creates a real arbitrage choice")
price_axes[0].axvspan(1, 3, color="tab:green", alpha=0.12, label="low-price charging opportunity")
price_axes[0].axvspan(4, 6, color="tab:red", alpha=0.10, label="high-price discharge opportunity")
price_axes[0].legend(loc="upper right")

price_bars = price_axes[1].bar(
    fcr_block_indices,
    fcr_prices,
    width=0.82,
    color="tab:orange",
)
price_axes[1].set_ylim(0, max(fcr_prices) * 1.35)
price_axes[1].set_ylabel("FCR price (EUR/kW/block)")
price_axes[1].set_xlabel("Four-hour FCR block")
price_axes[1].set_title("Synthetic input 2: every reserve block has positive capacity value")
annotate_bars(price_axes[1], price_bars, digits=2)

generated_figures["input_prices"] = price_figure
display(price_figure)
plt.close(price_figure)

## 2. Solve the restriction $\overline{P}$ and relaxation $\underline{P}$

The [reference paper](https://arxiv.org/abs/2510.10856) starts from the continuous-time robust problem (5). Theorem 1 gives its exact finite-dimensional mixed-binary bilinear reformulation $P$, comprising (8b), (8d), (9), and (10a)–(10k). The remaining bilinear constraint (10k) comes from the robust upper stored-energy bound. OEL keeps the paper's formulation notation with the following API mapping:

| OEL formulation | Paper notation | Construction and role |
| --- | --- | --- |
| restriction | $\overline{P}$ | Replace (10k) with the conservative mixed-binary linear constraints (14a)–(14d); candidate deployable plan. |
| exact | $P$ | Retain the bilinear constraint (10k); optional licensed validation evidence. |
| relaxation | $\underline{P}$ | Delete (10k) while retaining (10a)–(10j); optimistic bound evidence only. |

Section 4.2 of the paper interprets $\underline{P}$ and $\overline{P}$ as lower-convex and upper-concave McCormick envelopes of the bilinear region, respectively. They are not written using the textbook lifted-product/four-inequality presentation, but the McCormick-envelope interpretation is the authors' own. The paper minimizes cost, so its ordering is

$$
z_{\underline{P}}
\le
z_P
\le
z_{\overline{P}}.
$$

OEL reports the equivalent objective as maximized profit, reversing the formulation order:

$$
z_{\mathrm{restriction}}
\le
z_{\mathrm{exact}}
\le
z_{\mathrm{relaxation}}.
$$

All formulations receive exactly the same battery, price, time, terminal, uncertainty, cutoff, and numeric-policy inputs. Only $\overline{P}$ can emit a robust-labelled plan, and only after its independent deployability audit passes. Consequently, the relaxation reports deployability as *not_applicable*: this means that deployability lies outside its upper-bound-only role, not that a candidate restriction failed an audit. *blocked* is reserved for a restriction that was considered for deployment but failed one or more gates. Even when $\underline{P}$ and $\overline{P}$ happen to return identical decisions, the plan is emitted from the audited restriction result.


In [ ]:
restriction_result = solve_robust_fcr_bid(
    walkthrough_problem,
    formulation="restriction",
)
relaxation_result = solve_robust_fcr_bid(
    walkthrough_problem,
    formulation="relaxation",
)
bound_comparison = compare_robust_fcr_bounds(
    restriction_result,
    relaxation_result,
)

assert restriction_result.decisions is not None
assert relaxation_result.decisions is not None
assert restriction_result.deployable_plan is not None


In [ ]:
solver_table = pd.DataFrame(
    [
        {
            "formulation": result.formulation.kind,
            "role": result.formulation.deployability_role,
            "solve status": result.solver.solve_status,
            "termination": result.solver.termination_status,
            "objective (EUR)": result.solver.objective_value_eur,
            "best upper bound (EUR)": result.solver.best_upper_bound_eur,
            "relative gap": result.solver.relative_gap,
            "deployability status (restriction only)": result.deployability_status,
            "plan emitted": result.deployable_plan is not None,
            "solver": result.solver.solver_name,
            "solver version": result.solver.solver_version,
        }
        for result in (restriction_result, relaxation_result)
    ]
)
bound_table = pd.DataFrame(
    [bound_comparison.restriction_le_relaxation.to_dict()]
)
display(solver_table, bound_table)


In [ ]:
energy_schedule_table = pd.DataFrame(
    {
        "delivery hour": hour_indices,
        "restriction baseline (kW, charge-positive)": restriction_result.decisions.energy_baseline_charge_positive_kw,
        "relaxation baseline (kW, charge-positive)": relaxation_result.decisions.energy_baseline_charge_positive_kw,
    }
)
fcr_schedule_table = pd.DataFrame(
    {
        "four-hour block": fcr_block_indices,
        "restriction symmetric FCR (kW)": restriction_result.decisions.fcr_capacity_kw,
        "relaxation symmetric FCR (kW)": relaxation_result.decisions.fcr_capacity_kw,
    }
)
audit = restriction_result.feasibility_audit
assert audit is not None
audit_table = pd.DataFrame(
    [
        {
            "equation family": family.family_id,
            "relation": family.relation,
            "unit": family.canonical_unit,
            "checks": family.checked_count,
            "maximum positive violation": family.maximum_positive_violation,
        }
        for family in audit.constraint_families
    ]
)
audit_summary_table = pd.DataFrame(
    [
        ("maximum modeled slack", max(audit.modeled_slacks.values()), "canonical energy units"),
        ("maximum independent residual", audit_table["maximum positive violation"].max(), "family canonical unit"),
        ("deployability threshold", audit.deployability_threshold, "absolute per canonical unit"),
        ("passes deployability audit", audit.passes_deployability_audit, "boolean"),
    ],
    columns=["quantity", "value", "unit"],
)
display(energy_schedule_table, fcr_schedule_table, audit_summary_table)


In [ ]:
baseline_hourly = list(restriction_result.decisions.energy_baseline_charge_positive_kw)
reserve_by_hour = [
    restriction_result.decisions.fcr_capacity_kw[hour // 4]
    for hour in hour_indices
]
discharge_edge = [
    baseline - reserve
    for baseline, reserve in zip(baseline_hourly, reserve_by_hour, strict=True)
]
charge_edge = [
    baseline + reserve
    for baseline, reserve in zip(baseline_hourly, reserve_by_hour, strict=True)
]

schedule_figure, schedule_axis = plt.subplots(figsize=(11, 5.5), constrained_layout=True)
schedule_axis.fill_between(
    list(range(25)),
    [*discharge_edge, discharge_edge[-1]],
    [*charge_edge, charge_edge[-1]],
    step="post",
    alpha=0.24,
    color="tab:orange",
    label="accepted symmetric reserve band p ± r",
)
schedule_axis.step(
    list(range(25)),
    [*baseline_hourly, baseline_hourly[-1]],
    where="post",
    color="tab:blue",
    linewidth=2,
    label="hourly energy baseline p",
)
schedule_axis.axhline(
    battery.maximum_charge_power_kw,
    color="black",
    linestyle="--",
    linewidth=1,
    label="battery power limits",
)
schedule_axis.axhline(
    -battery.maximum_discharge_power_kw,
    color="black",
    linestyle="--",
    linewidth=1,
)
schedule_axis.axhline(0.0, color="black", linewidth=0.7)
schedule_axis.set_xlim(0, 24)
schedule_axis.set_ylim(-11.5, 11.5)
schedule_axis.set_xlabel("Delivery hour")
schedule_axis.set_ylabel("Battery power (kW, charge-positive)")
schedule_axis.set_title("Joint bid: arbitrage baseline and symmetric reserve must share converter headroom")
schedule_axis.legend(loc="lower right", ncol=2)

generated_figures["joint_bid_and_headroom"] = schedule_figure
display(schedule_figure)
plt.close(schedule_figure)


## 3. Explain why reserve is 3.636 kW, not 10 kW

Pointwise symmetric reserve must fit around the hourly baseline:

$$
-P^{\max}_{\mathrm{discharge}}
\le p_h-r_b,
\qquad
p_h+r_b
\le P^{\max}_{\mathrm{charge}}.
$$

The 10 kWh distance from the initial state to either energy boundary and the 2.75 h horizon budget also give the hand-calculable robust-energy scale

$$
r_{\mathrm{energy}}
=
\frac{\min(E_0-E^{\min},\,E^{\max}-E_0)}{\gamma}
=
\frac{10\ \mathrm{kWh}}{2.75\ \mathrm{h}}
=
3.636\ \mathrm{kW}.
$$

This quotient is an explanatory scale for this unit-efficiency case, not a replacement for the full mixed-binary formulation or its independent audit.

The 6.364 kW value plotted one hour after the first 3.636 kW value is not additional reserve: both are hourly day-ahead charging baselines, while the symmetric reserve remains fixed at 3.636 kW throughout the four-hour block. After the earlier high-price discharge, the battery is at its minimum stored energy and must recover 10 kWh to satisfy the terminal rule. With unit efficiency and one-hour energy intervals, the two charging baselines provide exactly

$$
3.636\ \mathrm{kW} \times 1\ \mathrm{h}
+
6.364\ \mathrm{kW} \times 1\ \mathrm{h}
=10\ \mathrm{kWh}.
$$

In the first hour, $p_h-r_b=3.636-3.636=0$ kW, so full activation toward discharge cannot draw energy from the empty battery. In the next hour, $p_h+r_b=6.364+3.636=10$ kW, so the opposite reserve edge reaches the charging-power limit. Because the later day-ahead prices are flat, the timing of this recharge is not uniquely determined economically; this is one solver-selected extreme-point schedule.


In [ ]:
planning_hours = [(index + 1) * 0.25 for index in range(96)]
baseline_by_planning = [
    restriction_result.decisions.energy_baseline_charge_positive_kw[
        time_contract.planning_to_energy[index]
    ]
    for index in range(96)
]
reserve_by_planning = [
    restriction_result.decisions.fcr_capacity_kw[
        time_contract.planning_to_fcr[index]
    ]
    for index in range(96)
]
headroom_table = pd.DataFrame(
    {
        "planning hour end": planning_hours,
        "discharge edge p-r (kW)": [
            p - r for p, r in zip(baseline_by_planning, reserve_by_planning, strict=True)
        ],
        "discharge limit (kW)": [-battery.maximum_discharge_power_kw] * 96,
        "charge edge p+r (kW)": [
            p + r for p, r in zip(baseline_by_planning, reserve_by_planning, strict=True)
        ],
        "charge limit (kW)": [battery.maximum_charge_power_kw] * 96,
    }
)

robust_energy_capacity_kw = min(
    battery.initial_stored_energy_kwh - battery.minimum_stored_energy_kwh,
    battery.maximum_stored_energy_kwh - battery.initial_stored_energy_kwh,
) / horizon_budget.horizon_activation_budget_hours

attribution_rows = []
for block in fcr_block_indices:
    block_baseline = baseline_hourly[4 * block : 4 * block + 4]
    discharge_headroom = min(
        value + battery.maximum_discharge_power_kw
        for value in block_baseline
    )
    charge_headroom = min(
        battery.maximum_charge_power_kw - value
        for value in block_baseline
    )
    pointwise_capacity = min(discharge_headroom, charge_headroom)
    selected_capacity = restriction_result.decisions.fcr_capacity_kw[block]
    if pointwise_capacity == 0.0:
        binding_reason = "arbitrage consumes pointwise power"
    elif abs(pointwise_capacity - robust_energy_capacity_kw) <= 1e-9:
        binding_reason = "power and robust-energy limits coincide"
    else:
        binding_reason = "robust stored-energy protection"
    attribution_rows.append(
        {
            "FCR block": block,
            "pointwise power-only capacity (kW)": pointwise_capacity,
            "robust-energy scale (kW)": robust_energy_capacity_kw,
            "selected symmetric FCR (kW)": selected_capacity,
            "interpretation": binding_reason,
        }
    )

reserve_attribution_table = pd.DataFrame(attribution_rows)
display(reserve_attribution_table, audit_table)


### Reading “capacity allowed by pointwise power alone”

Here, *pointwise* means that the converter power limits are checked independently around each hourly baseline $p_h$. Because one symmetric reserve bid $r_b$ applies throughout a four-hour block $b$, it must fit in both directions during every hour in that block:

$$
r_b^{\mathrm{power}}
=
\min_{h\in b}
\left\{
p_h+P_{\mathrm{discharge}}^{\max},\,
P_{\mathrm{charge}}^{\max}-p_h
\right\}.
$$

This is a diagnostic power-only upper bound, not the selected reserve: it ignores stored energy and cumulative activation. It is 0 kW in blocks 0–1 because arbitrage reaches a converter limit, 3.636 kW in block 2 because the 6.364 kW charging baseline leaves only $10-6.364=3.636$ kW of upward charging headroom, and 10 kW in blocks 3–5 because their baseline is zero. The bidder nevertheless selects only 3.636 kW in those later blocks because the robust stored-energy constraint is tighter than their instantaneous power headroom.


In [ ]:
positions = fcr_block_indices
width = 0.36
robust_figure, robust_axis = plt.subplots(figsize=(10.5, 5.5), constrained_layout=True)
power_bars = robust_axis.bar(
    [value - width / 2 for value in positions],
    reserve_attribution_table["pointwise power-only capacity (kW)"],
    width=width,
    color="tab:blue",
    alpha=0.75,
    label="capacity allowed by pointwise power alone",
)
selected_bars = robust_axis.bar(
    [value + width / 2 for value in positions],
    reserve_attribution_table["selected symmetric FCR (kW)"],
    width=width,
    color="tab:orange",
    label="restriction-selected robust FCR",
)
robust_axis.axhline(
    robust_energy_capacity_kw,
    color="tab:red",
    linestyle="--",
    label=f"10 kWh / 2.75 h = {robust_energy_capacity_kw:.3f} kW",
)
robust_axis.set_xticks(positions)
robust_axis.set_xlabel("Four-hour FCR block")
robust_axis.set_ylabel("Symmetric capacity (kW)")
robust_axis.set_ylim(0, 11.2)
robust_axis.set_title("Constraint attribution: available converter power is not always robustly usable")
robust_axis.legend(loc="upper left")
annotate_bars(robust_axis, selected_bars)

generated_figures["reserve_constraint_attribution"] = robust_figure
display(robust_figure)
plt.close(robust_figure)


## 4. Keep the ex-ante objective separate from realized profit

$z$ is the objective value, measured in EUR, that Feature 013 uses to rank feasible combinations of the day-ahead baseline and FCR-capacity bid. A larger $z$ represents a more valuable plan under the prices and information available when the bid is selected. In the reference paper's discharge-positive notation,

$$
z
=
\underbrace{
\sum_h \Delta t\,
\lambda_h^{\mathrm{DA}}x_h^{0,\mathrm{source}}
}_{\text{day-ahead baseline value}}
+
\underbrace{
\sum_b \lambda_b^{\mathrm{FCR}}x_b^r
}_{\text{FCR capacity value}}
-
\underbrace{
c^{\mathrm{slack}}\sum_k s_k
}_{\text{feasibility-slack penalty}}.
$$

The day-ahead term values scheduled energy: $\lambda_h^{\mathrm{DA}}$ is in EUR/kWh, power is multiplied by the interval duration $\Delta t$, and $x_h^{0,\mathrm{source}}>0$ denotes a sale through discharge. Since OEL's public baseline is charge-positive, $x_h^{0,\mathrm{source}}=-p_h^{\mathrm{OEL}}$; charging is therefore an energy cost and discharging is revenue. The FCR term pays for reserved symmetric capacity whether or not it is activated. Its input price is already expressed in EUR/kW/block, so no additional block-duration multiplier appears. The final term penalizes any modeled energy slack; an eligible restriction plan must ultimately have no material slack.

For the teaching case, both formulations report

$$
z
=
2.800000
+
1.454545
-
0
=
4.254545\ \mathrm{EUR}.
$$

This is an *ex-ante optimization score*, not realized operating profit. It excludes the still-unknown activation energy, tracking error, saturation, and settlement outcomes. Those quantities are calculated ex post only after Feature 012 receives the accepted plan and a realized activation trace.


In [ ]:
objective_rows = []
for result in (restriction_result, relaxation_result):
    objective = result.raw_objective
    assert objective is not None
    objective_rows.extend(
        [
            (result.formulation.kind, "day-ahead energy value", objective.day_ahead_energy_value_eur, "EUR"),
            (result.formulation.kind, "FCR capacity value", objective.fcr_capacity_value_eur, "EUR"),
            (result.formulation.kind, "feasibility slack penalty", -objective.feasibility_slack_penalty_eur, "EUR"),
            (result.formulation.kind, "total objective", objective.total_objective_eur, "EUR"),
            (result.formulation.kind, "reconciliation residual", objective.reconciliation_residual_eur, "EUR"),
        ]
    )
objective_table = pd.DataFrame(
    objective_rows,
    columns=["formulation", "component", "value", "unit"],
)
objective_table


In [ ]:
restriction_objective = restriction_result.raw_objective
objective_components = pd.DataFrame(
    {
        "component": ["day-ahead arbitrage", "FCR capacity", "slack penalty"],
        "value (EUR)": [
            restriction_objective.day_ahead_energy_value_eur,
            restriction_objective.fcr_capacity_value_eur,
            -restriction_objective.feasibility_slack_penalty_eur,
        ],
    }
)
objective_figure, objective_axis = plt.subplots(figsize=(9, 4.5), constrained_layout=True)
objective_bars = objective_axis.bar(
    objective_components["component"],
    objective_components["value (EUR)"],
    color=["tab:blue", "tab:orange", "tab:red"],
)
objective_axis.axhline(0.0, color="black", linewidth=0.8)
objective_axis.axhline(
    restriction_objective.total_objective_eur,
    color="tab:green",
    linestyle="--",
    label=f"total ex-ante objective = {restriction_objective.total_objective_eur:.3f} EUR",
)
objective_axis.set_ylabel("Objective contribution (EUR)")
objective_axis.set_title("The selected plan earns from both arbitrage and reserve")
objective_axis.legend()
annotate_bars(objective_axis, objective_bars)

generated_figures["ex_ante_objective_components"] = objective_figure
display(objective_figure)
plt.close(objective_figure)


## 5. Post-acceptance stress replay through Feature 012

The bid is now fixed. Three evaluation-only traces are attached to the unchanged restriction plan:

- **nominal**: no activation;
- **upward pulses**: full upward activation for one quarter-hour every 2.25 h after reserve begins;
- **downward pulses**: the same timing in the opposite direction.

Each stress trace contains 2.0 activated hours, below the 2.75 h horizon budget. The pattern is an interpretable recovery-spaced stress trajectory; a single realized trace is not proof of the entire robust counterpart. Crucially, none of these samples existed in the Feature 013 problem.

Feature 012 alone applies

$$
p_k^{\mathrm{requested}}=p_{h(k)}-a_k r_{b(k)}
$$

and the common battery transition to produce delivered power, stored energy, tracking, saturation, settlement, and realized profit.


### Step 1 — Construct the post-acceptance requests

The next cell does not optimize or change the bid. It takes the already accepted restriction plan and packages everything Feature 012 needs for evaluation: the 15-minute time grid, one of the three synthetic activation traces, the price context, and a battery component with the same physical parameters. It creates one request per scenario while keeping the plan identical in all three.


In [ ]:
def build_walkthrough_evaluation_request(plan, activation_case):
    period = plan.energy_schedule.covered_period
    evaluation_time = MultiRateTimeContract(
        start_utc=period.start_utc,
        end_utc=period.end_utc,
        activation_resolution="PT15M",
        state_checkpoint_resolution="PT15M",
        energy_resolution="PT1H",
        fcr_resolution="PT4H",
        market_timezone="Europe/Paris",
        interval_convention="start-inclusive/end-exclusive",
    )
    sample_count = len(evaluation_time.index.activation_intervals)
    if activation_case == "nominal":
        activation_values = (0.0,) * sample_count
    elif activation_case == "upward pulses":
        activation_values = tuple(
            1.0 if index >= 32 and (index - 32) % 9 == 0 else 0.0
            for index in range(sample_count)
        )
    elif activation_case == "downward pulses":
        activation_values = tuple(
            -1.0 if index >= 32 and (index - 32) % 9 == 0 else 0.0
            for index in range(sample_count)
        )
    else:
        raise ValueError(f"unsupported activation case: {activation_case}")

    case_id = activation_case.replace(" ", "-")
    activation_request = ActivationImportRequest(
        source_name=f"Feature 013 walkthrough {activation_case}",
        source_locator=(
            "examples/robust_fcr_bidding/"
            "robust_fcr_bidding_walkthrough.ipynb"
        ),
        source_revision="sha256:" + stable_fingerprint(list(activation_values)),
        source_timezone="Europe/Paris",
        requested_start_utc=period.start_utc,
        requested_end_utc=period.end_utc,
        source_resolution="PT15M",
        source_value_kind="interval_average",
        source_sign_meaning="positive source activation means upward/discharge",
        normalization_rule={
            "conversion": "identity",
            "formula": "activation = source_value",
            "point_to_interval": "already_interval_average",
            "clip_min": -1.0,
            "clip_max": 1.0,
            "target_sign": "positive is upward/discharge",
        },
        redistribution_basis={
            "source": "project-authored notebook stress input",
            "terms": "Apache-2.0",
        },
    )
    activation_trace = import_regulation_activation_trace(
        activation_request,
        [
            {
                "start_utc": interval.start_utc,
                "end_utc": interval.end_utc,
                "value": value,
                "source_value": value,
                "source_unit": "dimensionless",
            }
            for interval, value in zip(
                evaluation_time.index.activation_intervals,
                activation_values,
                strict=True,
            )
        ],
        trace_id=f"feature-013-walkthrough-{case_id}-v2",
        created_at_utc=datetime(2026, 8, 31, 12, tzinfo=UTC),
    )
    day_prices_eur_per_mwh = tuple(
        value * 1000.0
        for value in walkthrough_problem.prices.day_ahead_energy_prices_eur_per_kwh
    )
    day_observations = tuple(
        EvaluationPriceObservation(
            interval=item.interval,
            price_kind="day_ahead_energy",
            source_value=price,
            source_unit="EUR/MWh",
            canonical_value=price,
            canonical_unit="EUR/MWh",
            conversion_id="walkthrough-eur-per-kwh-to-eur-per-mwh-v1",
            source_ref=f"feature_013_walkthrough_day_ahead:{index}",
        )
        for index, (item, price) in enumerate(
            zip(plan.energy_schedule.intervals, day_prices_eur_per_mwh, strict=True)
        )
    )
    fcr_observations = tuple(
        EvaluationPriceObservation(
            interval=item.interval,
            price_kind="fcr_capacity",
            source_value=price,
            source_unit="EUR/kW/block",
            canonical_value=price,
            canonical_unit="EUR/kW/block",
            conversion_id="walkthrough-identity-v1",
            source_ref=item.price_observation_ref,
        )
        for item, price in zip(
            plan.fcr_schedule.intervals,
            walkthrough_problem.prices.fcr_capacity_prices_eur_per_kw_block,
            strict=True,
        )
    )
    accepted = {
        item.volume_observation_ref: {
            "capacity_kw": item.capacity_kw,
            "unit": "kW",
            "evidence_kind": "authored_post_acceptance",
        }
        for item in plan.fcr_schedule.intervals
    }
    market_context = EvaluationMarketContext(
        context_kind="reference_result",
        semantics_version="feature-013-walkthrough-market-v2",
        period=CanonicalInterval.from_utc(period.start_utc, period.end_utc),
        day_ahead_prices=day_observations,
        fcr_capacity_prices=fcr_observations,
        accepted_bid_evidence=accepted,
        source_fingerprints={
            "post_acceptance_market": stable_fingerprint(
                {
                    "plan": plan.plan_fingerprint,
                    "day_prices_eur_per_mwh": day_prices_eur_per_mwh,
                    "fcr_prices": walkthrough_problem.prices.fcr_capacity_prices_eur_per_kw_block,
                }
            )
        },
    )
    replay_battery = BatteryComponent(
        name="feature-013-walkthrough-battery",
        capacity_kwh=battery.maximum_stored_energy_kwh,
        max_charge_kw=battery.maximum_charge_power_kw,
        max_discharge_kw=battery.maximum_discharge_power_kw,
        charge_efficiency=battery.charge_efficiency,
        discharge_efficiency=battery.discharge_efficiency,
        initial_soc=battery.initial_stored_energy_kwh / battery.maximum_stored_energy_kwh,
        min_soc=battery.minimum_stored_energy_kwh / battery.maximum_stored_energy_kwh,
        max_soc=1.0,
    )
    return FCRBidEvaluationRequest(
        evaluation_id=f"feature-013-walkthrough-{case_id}-v2",
        plan=plan,
        market_context=market_context,
        activation_trace=activation_trace,
        battery=replay_battery,
        time_contract=evaluation_time,
        settlement_profile="robust-storage-bidding-reference-settlement-v1",
        penalty_rule=None,
        terminal_soc_target=None,
        comparison_fixture=None,
        numeric_tolerance=1e-6,
    )

activation_cases = ("nominal", "upward pulses", "downward pulses")
walkthrough_evaluation_requests = {
    case: build_walkthrough_evaluation_request(
        restriction_result.deployable_plan,
        case,
    )
    for case in activation_cases
}
walkthrough_evaluation_request = walkthrough_evaluation_requests["upward pulses"]


### Step 2 — Replay each request through Feature 012

The next cell evaluates the three fixed-plan requests independently. For every 15-minute interval, Feature 012 combines baseline and activation, applies the battery transition, and records requested and delivered power, stored energy, tracking error, saturation, and settlement. The assertion confirms that all three evaluations completed; it does not compare or re-optimize their bids.


In [ ]:
walkthrough_evaluation_reports = {
    case: evaluate_fixed_fcr_bid(request)
    for case, request in walkthrough_evaluation_requests.items()
}
walkthrough_evaluation_payloads = {
    case: report.to_dict()
    for case, report in walkthrough_evaluation_reports.items()
}
assert all(
    report.status == "settled"
    for report in walkthrough_evaluation_reports.values()
)

walkthrough_evaluation_report = walkthrough_evaluation_reports["upward pulses"]
walkthrough_evaluation_payload = walkthrough_evaluation_payloads["upward pulses"]


### Step 3 — Summarize the replay evidence

The next cell reshapes the evaluation reports into three views: an activation-budget summary, one physical summary per scenario, and a detailed 15-minute trace. Only the first 40 rows of the upward-pulse trace are displayed so that the transition into the reserve period at hour 8 can be inspected without printing all 288 scenario-samples.


In [ ]:
replay_rows = []
physical_summary_rows = []
activation_summary_rows = []
for case in activation_cases:
    payload = walkthrough_evaluation_payloads[case]
    samples = payload["physical_trace"]["samples"]
    for index, item in enumerate(samples):
        replay_rows.append(
            {
                "scenario": case,
                "sample": index,
                "hours since start": index * 0.25,
                "activation": item["activation"],
                "baseline power (kW)": item["baseline_power_kw"],
                "requested combined power (kW)": item["requested_combined_power_kw"],
                "delivered combined power (kW)": item["delivered_combined_power_kw"],
                "stored energy after (kWh)": item["stored_energy_after_kwh"],
                "absolute tracking error (kW)": item["absolute_tracking_error_kw"],
                "saturation": ", ".join(item["saturation_causes"]) or "none",
            }
        )
    metrics = payload["physical_metrics"]
    physical_summary_rows.append(
        {
            "scenario": case,
            "requested minimum (kW)": metrics["requested_power_kw"]["minimum"],
            "requested maximum (kW)": metrics["requested_power_kw"]["maximum"],
            "stored-energy minimum (kWh)": metrics["stored_energy_kwh"]["minimum"],
            "stored-energy maximum (kWh)": metrics["stored_energy_kwh"]["maximum"],
            "terminal stored energy (kWh)": metrics["stored_energy_kwh"]["terminal"],
            "throughput (kWh)": metrics["throughput_kwh"],
            "tracking-error sum (kW-samples)": metrics["absolute_tracking_error_kw_sum"],
            "saturation samples": metrics["saturation_sample_count"],
        }
    )
    values = [
        sample["activation"]
        for sample in samples
    ]
    activation_summary_rows.append(
        {
            "scenario": case,
            "nonzero quarter-hours": sum(value != 0.0 for value in values),
            "absolute activated hours": 0.25 * sum(abs(value) for value in values),
            "horizon budget (h)": horizon_budget.horizon_activation_budget_hours,
        }
    )

replay_trace_table = pd.DataFrame(replay_rows)
physical_summary_table = pd.DataFrame(physical_summary_rows)
activation_stress_table = pd.DataFrame(activation_summary_rows)
physical_metrics = walkthrough_evaluation_payload["physical_metrics"]
display(
    activation_stress_table,
    physical_summary_table,
    replay_trace_table[replay_trace_table["scenario"] == "upward pulses"].head(40),
)


### How to read the physical-replay figure

The three panels show the same accepted plan under different information revealed after acceptance:

1. **Activation:** the red $+1$ pulses request upward regulation, meaning discharge; the green $-1$ pulses request downward regulation, meaning charge. The nominal zero signal is omitted from this panel because it would lie on the horizontal axis.
2. **Requested battery power:** Feature 012 applies $p_k^{\mathrm{requested}}=p_{h(k)}-a_k r_{b(k)}$. OEL power is charge-positive, so upward activation moves the red curve downward and downward activation moves the green curve upward. The scenarios overlap before reserve begins at hour 8. The dotted lines are the $\pm10$ kW converter limits. Requested and delivered power coincide here because no scenario saturates.
3. **Stored energy:** each curve starts at 10 kWh and integrates its scenario's delivered power. Upward pulses consume more stored energy than the nominal case, while downward pulses add more. The dotted 0 and 20 kWh lines are the energy limits; all three trajectories remain inside them.

The figure therefore demonstrates physical feasibility for these three authored traces. It is an interpretable stress replay, not a proof for every signal in the uncertainty set; the robust formulation and independent audit provide that broader evidence.


In [ ]:
scenario_colors = {
    "nominal": "black",
    "upward pulses": "tab:red",
    "downward pulses": "tab:green",
}
replay_figure, replay_axes = plt.subplots(3, 1, figsize=(11, 9), constrained_layout=True)

for case in ("upward pulses", "downward pulses"):
    case_table_for_plot = replay_trace_table[
        replay_trace_table["scenario"] == case
    ]
    activation_values_for_plot = case_table_for_plot["activation"].tolist()
    replay_axes[0].step(
        [0.25 * index for index in range(97)],
        [*activation_values_for_plot, activation_values_for_plot[-1]],
        where="post",
        color=scenario_colors[case],
        label=case,
    )
replay_axes[0].set_ylabel("Activation (dimensionless)")
replay_axes[0].set_xlabel("Hours since delivery start")
replay_axes[0].set_title("Evaluation-only activation: one full pulse every 2.25 h after hour 8")
replay_axes[0].set_ylim(-1.15, 1.15)
replay_axes[0].set_xlim(0, 24)
replay_axes[0].legend()

for case in activation_cases:
    case_table_for_plot = replay_trace_table[
        replay_trace_table["scenario"] == case
    ]
    requested_values_for_plot = case_table_for_plot["requested combined power (kW)"].tolist()
    replay_axes[1].step(
        [0.25 * index for index in range(97)],
        [*requested_values_for_plot, requested_values_for_plot[-1]],
        where="post",
        color=scenario_colors[case],
        linestyle="--" if case == "nominal" else "-",
        label=case,
    )
replay_axes[1].axhline(10.0, color="grey", linestyle=":", label="battery power limits")
replay_axes[1].axhline(-10.0, color="grey", linestyle=":")
replay_axes[1].set_ylabel("Requested power (kW, charge-positive)")
replay_axes[1].set_xlabel("Hours since delivery start")
replay_axes[1].set_title("Feature 012 requests remain within the converter limits")
replay_axes[1].set_xlim(0, 24)
replay_axes[1].legend(ncol=2)

for case in activation_cases:
    case_table_for_plot = replay_trace_table[
        replay_trace_table["scenario"] == case
    ]
    replay_axes[2].plot(
        [0.0, *[0.25 * (index + 1) for index in range(96)]],
        [
            battery.initial_stored_energy_kwh,
            *case_table_for_plot["stored energy after (kWh)"].tolist(),
        ],
        color=scenario_colors[case],
        linestyle="--" if case == "nominal" else "-",
        label=case,
    )
replay_axes[2].axhline(0.0, color="grey", linestyle=":", label="stored-energy limits")
replay_axes[2].axhline(20.0, color="grey", linestyle=":")
replay_axes[2].set_ylabel("Stored energy (kWh)")
replay_axes[2].set_xlabel("Hours since delivery start")
replay_axes[2].set_title("The fixed plan remains feasible under both unseen pulse directions")
replay_axes[2].legend(ncol=2)

generated_figures["post_acceptance_stress_replay"] = replay_figure
display(replay_figure)
plt.close(replay_figure)


## 6. Realized settlement is scenario-dependent

The ex-ante optimization objective is one number because all three replays use the same fixed bid. Feature 012 settlement changes with realized activation. Keeping these quantities separate prevents an optimization score from being mislabeled as realized profit.

The accounting below follows the realized-profit calculation used for the numerical backtests in Section 6 of the [reference paper](https://arxiv.org/abs/2510.10856) and made explicit in its accompanying [`bt_simulation`](../../baselines/robust_storage_bidding/code/03_backtest_save.jl) implementation. In OEL's charge-positive convention, the reference-result settlement profile evaluates

$$
\pi^{\mathrm{DA}}
=-\sum_h p_h\,\Delta t_h\,\lambda_h^{\mathrm{DA}},
\qquad
\pi^{\mathrm{FCR,cap}}
=\sum_b r_b\,\lambda_b^{\mathrm{FCR}},
$$

$$
\pi^{\mathrm{activation}}
=-\sum_k p_k^{\mathrm{FCR,del}}\,\Delta t_k\,\lambda_{h(k)}^{\mathrm{DA}},
\qquad
\pi^{\mathrm{realized}}
=\pi^{\mathrm{DA}}+\pi^{\mathrm{FCR,cap}}+\pi^{\mathrm{activation}}-\pi^{\mathrm{penalty}}.
$$

Here $p_h$ is the accepted hourly baseline, $r_b$ is the accepted symmetric capacity, and $p_k^{\mathrm{FCR,del}}$ is the FCR component actually delivered by the battery. Negative delivered FCR power is upward regulation through discharge and therefore earns activation revenue; positive delivered FCR power is downward regulation through charging and therefore creates an energy cost. OEL pays only delivered activation energy. In these three traces requested and delivered activation coincide because there is no tracking error or saturation.

This is a reproduction of the reference study's accounting convention, in which delivered activation energy is valued at the corresponding day-ahead price. It is not a claim that this synthetic example reproduces the settlement rules of a particular real FCR market. Here *realized* means that the accepted bid is evaluated using an activation trace that becomes known only after bid selection.


In [ ]:
settlement_components = (
    "day_ahead_revenue",
    "fcr_capacity_revenue",
    "activation_energy_revenue",
    "penalty",
    "intraday",
    "imbalance",
    "degradation",
    "terminal_value",
)
settlement_rows = []
economics_rows = []
for case in activation_cases:
    settlement = walkthrough_evaluation_payloads[case]["settlement"]
    for component in settlement_components:
        evidence = settlement[component]
        settlement_rows.append(
            {
                "scenario": case,
                "component": component,
                "status": evidence["status"],
                "value (EUR)": evidence["value_eur"],
                "reason": evidence["reason"],
            }
        )
    economics_rows.append(
        {
            "scenario": case,
            "Feature 013 ex-ante objective (EUR)": restriction_result.solver.objective_value_eur,
            "Feature 013 best upper bound (EUR)": restriction_result.solver.best_upper_bound_eur,
            "Feature 012 realized profit (EUR)": settlement["total_profit_eur"],
            "settlement reconciliation error (EUR)": settlement["reconciliation_error_eur"],
        }
    )

settlement_table = pd.DataFrame(settlement_rows)
economics_separation_table = pd.DataFrame(economics_rows)
settlement_payload = walkthrough_evaluation_payload["settlement"]
display(settlement_table, economics_separation_table)


### Reading the realized-settlement figure

For the authored case, only seven hourly baseline positions are nonzero. Their day-ahead settlement is

$$
\pi^{\mathrm{DA}}
=10(0.03)-10(0.02)-10(0.01)+10(0.18)+10(0.16)-3.636364(0.06)-6.363636(0.06)
=2.800000\ \mathrm{EUR}.
$$

The positive terms are sales through discharge and the negative terms are purchases for charging. Four active FCR blocks each offer $3.636364$ kW at $0.10$ EUR/kW/block, giving

$$
\pi^{\mathrm{FCR,cap}}
=4\times3.636364\times0.10
=1.454545\ \mathrm{EUR}.
$$

Each pulse trace contains eight full-activation quarter-hours, hence $8\times0.25\times3.636364=7.272727$ kWh of delivered directional FCR energy. All pulses occur after hour 8, where the synthetic day-ahead price is $0.06$ EUR/kWh. Their activation value is therefore

$$
7.272727\times0.06=0.436364\ \mathrm{EUR}.
$$

The upper panel decomposes these values: nominal activation gives $2.800000+1.454545=4.254545$ EUR; upward/discharge pulses add $0.436364$ EUR and give $4.690909$ EUR; downward/charge pulses subtract $0.436364$ EUR and give $3.818182$ EUR. The lower panel compares those scenario-dependent realized profits with the unchanged $4.254545$ EUR ex-ante objective.

The penalty is zero because no penalty rule is configured and no energy is undelivered. Intraday, imbalance, degradation, and terminal-value components are unavailable in this profile and are excluded rather than silently treated as zero.


In [ ]:
available_components = (
    "day_ahead_revenue",
    "fcr_capacity_revenue",
    "activation_energy_revenue",
    "penalty",
)
settlement_figure, settlement_axes = plt.subplots(2, 1, figsize=(12.5, 7.5), constrained_layout=True)
bottoms = [0.0] * len(activation_cases)
for component in available_components:
    values = []
    for case in activation_cases:
        rows = settlement_table[
            (settlement_table["scenario"] == case)
            & (settlement_table["component"] == component)
        ]
        value = rows.iloc[0]["value (EUR)"]
        values.append(0.0 if value is None else float(value))
    bars = settlement_axes[0].bar(
        activation_cases,
        values,
        bottom=bottoms,
        label=component.replace("_", " "),
    )
    bottoms = [
        bottom + value
        for bottom, value in zip(bottoms, values, strict=True)
    ]
settlement_axes[0].axhline(0.0, color="black", linewidth=0.8)
settlement_axes[0].set_ylabel("Realized component value (EUR)")
settlement_axes[0].set_title("Feature 012 settlement components by activation scenario")
settlement_axes[0].legend(
    loc="center left",
    bbox_to_anchor=(1.01, 0.5),
    frameon=False,
)

objective_values = economics_separation_table[
    "Feature 013 ex-ante objective (EUR)"
].tolist()
profit_values = economics_separation_table[
    "Feature 012 realized profit (EUR)"
].tolist()
comparison_positions = list(range(len(activation_cases)))
comparison_width = 0.36
settlement_axes[1].bar(
    [position - comparison_width / 2 for position in comparison_positions],
    objective_values,
    width=comparison_width,
    label="Feature 013 ex-ante objective",
    color="tab:blue",
)
profit_bars = settlement_axes[1].bar(
    [position + comparison_width / 2 for position in comparison_positions],
    profit_values,
    width=comparison_width,
    label="Feature 012 realized profit",
    color="tab:green",
)
settlement_axes[1].set_xticks(comparison_positions, activation_cases)
settlement_axes[1].set_ylabel("Value (EUR)")
settlement_axes[1].set_title("One selected bid, different ex-post outcomes")
settlement_axes[1].legend(
    loc="center left",
    bbox_to_anchor=(1.01, 0.5),
    frameon=False,
)
annotate_bars(settlement_axes[1], profit_bars)

generated_figures["realized_economics_by_scenario"] = settlement_figure
display(settlement_figure)
plt.close(settlement_figure)


## 7. Reproducibility and lineage

Problem, solver result, decision vector, plan, activation, physical trace, settlement, and evaluation result have distinct semantic fingerprints. The activation-dependent fingerprints change by scenario; the pre-activation problem and plan fingerprints do not.


In [ ]:
fingerprint_rows = [
    ("shared", "bidding problem", walkthrough_problem.problem_fingerprint),
    ("shared", "restriction semantic result", restriction_result.semantic_result_fingerprint),
    ("shared", "relaxation semantic result", relaxation_result.semantic_result_fingerprint),
    ("shared", "bound comparison", bound_comparison.comparison_fingerprint),
    ("shared", "restriction decision vector", restriction_result.decisions.decision_vector_fingerprint),
    ("shared", "deployable plan", restriction_result.deployable_plan.plan_fingerprint),
    ("shared", "energy schedule", restriction_result.deployable_plan.energy_schedule.schedule_fingerprint),
    ("shared", "FCR schedule", restriction_result.deployable_plan.fcr_schedule.schedule_fingerprint),
]
for case in activation_cases:
    request = walkthrough_evaluation_requests[case]
    report = walkthrough_evaluation_reports[case]
    fingerprint_rows.extend(
        [
            (case, "activation trace", request.activation_trace.artifact_fingerprint),
            (case, "physical trace", report.trace_fingerprint),
            (case, "settlement", report.settlement_fingerprint),
            (case, "evaluation result", report.result_fingerprint),
        ]
    )

fingerprint_table = pd.DataFrame(
    fingerprint_rows,
    columns=["scenario", "artifact", "semantic fingerprint"],
)
fingerprint_table


## 8. Bounded Julia/Gurobi reference evidence

The default notebook inspects the normalized 2023-06-18 restriction fixture offline. It does not compare that historical schedule with the authored teaching case: the batteries, efficiencies, prices, terminal rule, and purpose differ.

The fixture records 24 hourly source-frame decisions, six FCR decisions, objective and bound values, provenance, and a source-precision envelope. It does not record unrounded decisions, formal termination, exact gap, residuals, tolerances, or a complete physical trace. Those fields remain unavailable rather than inferred.


In [ ]:
reference_fixture = load_robust_fcr_reference_fixture(
    REFERENCE_FIXTURE_PATH
)
reference_decision_intervals = build_reference_decision_intervals(
    reference_fixture
)


In [ ]:
reference_provenance_table = pd.DataFrame(
    [
        ("fixture id", reference_fixture.fixture_id),
        ("fixture fingerprint", reference_fixture.fixture_fingerprint),
        ("reference revision", reference_fixture.reference_provenance["reference_revision"]),
        ("raw result SHA-256", reference_fixture.reference_provenance["original_result_sha256"]),
        ("raw result bytes", reference_fixture.reference_provenance["original_result_byte_count"]),
        ("recorded objective (EUR)", reference_fixture.recorded_result.objective_value_eur),
        ("recorded bound (EUR)", reference_fixture.recorded_result.objective_bound_eur),
        ("saved decimal places", reference_fixture.recorded_result.saved_decimal_places),
    ],
    columns=["field", "value"],
)
reference_precision_table = pd.DataFrame(
    [
        ("decision half-unit", reference_fixture.precision_policy.decision_half_unit_kw, "kW"),
        ("objective half-unit", reference_fixture.precision_policy.objective_half_unit_eur, "EUR"),
        ("deployability threshold", reference_fixture.precision_policy.deployability_threshold, "absolute canonical unit"),
        ("interval closure", reference_fixture.precision_policy.interval_closure, "category"),
        ("certification method", reference_fixture.precision_policy.certification_method, "category"),
    ],
    columns=["quantity", "value", "unit"],
)
reference_unavailable_table = pd.DataFrame(
    [item.to_dict() for item in reference_fixture.unavailable_fields]
)
reference_decision_table = pd.DataFrame(
    [item.to_dict() for item in reference_decision_intervals]
)
display(
    reference_provenance_table,
    reference_precision_table,
    reference_unavailable_table,
    reference_decision_table,
)


In [ ]:
source_energy = reference_decision_table[
    reference_decision_table["decision_kind"] == "energy_x0"
]
source_fcr = reference_decision_table[
    reference_decision_table["decision_kind"] == "fcr_xr"
]
reference_precision_resolution_table = pd.DataFrame(
    [
        {
            "decision family": "hourly source x0",
            "count": len(source_energy),
            "unit": "kW, source discharge-positive",
            "saved decimals": reference_fixture.recorded_result.saved_decimal_places,
            "closed half-width": reference_fixture.precision_policy.decision_half_unit_kw,
            "meaning": "each displayed value denotes [value - 0.0005, value + 0.0005]",
        },
        {
            "decision family": "four-hour source xr",
            "count": len(source_fcr),
            "unit": "kW symmetric FCR",
            "saved decimals": reference_fixture.recorded_result.saved_decimal_places,
            "closed half-width": reference_fixture.precision_policy.decision_half_unit_kw,
            "meaning": "each displayed value denotes [value - 0.0005, value + 0.0005]",
        },
    ]
)
reference_precision_resolution_table


## Optional full native 2023-06-18 benchmark and side-by-side plot

Set RUN_NATIVE_REFERENCE_BENCHMARK to True only for an explicit research run. The complete historical problem can consume the full configured time limit. If a native decision vector is returned, the optional figure shows native and recorded Julia decisions in matching source units and places the numerical residual beside the explicit $\pm0.0005$ kW source-precision band.

Source-precision compatibility is strong bounded reproduction evidence. It is still not exact Julia parity without unrounded source decisions, common feasibility, and sufficient optimality evidence.


In [ ]:
native_reference_benchmark = {
    "status": "not_requested",
    "reason": "RUN_NATIVE_REFERENCE_BENCHMARK is False",
}
native_reference_comparison_table = pd.DataFrame()
benchmark_result = None
if RUN_NATIVE_REFERENCE_BENCHMARK:
    benchmark_policy = RobustFCRSolvePolicy(
        solver_name="highs",
        time_limit_seconds=REFERENCE_BENCHMARK_TIME_LIMIT_SECONDS,
        threads=1,
        random_seed=13,
        primal_feasibility_tolerance=1e-7,
        dual_feasibility_tolerance=1e-7,
        mip_feasibility_tolerance=1e-6,
        relative_mip_gap=1e-6,
        objective_comparison_absolute_tolerance_eur=1e-6,
        objective_comparison_relative_tolerance=1e-9,
        allow_feasible_incumbent_plan=False,
    )
    benchmark_problem = build_robust_fcr_reference_problem(
        reference_fixture,
        solve_policy=benchmark_policy,
    )
    benchmark_result = solve_robust_fcr_bid(
        benchmark_problem,
        formulation="restriction",
    )
    native_reference_benchmark = {
        "status": benchmark_result.solver.solve_status,
        "termination": benchmark_result.solver.termination_status,
        "objective_eur": benchmark_result.solver.objective_value_eur,
        "best_upper_bound_eur": benchmark_result.solver.best_upper_bound_eur,
        "deployability": benchmark_result.deployability_status,
        "result_fingerprint": benchmark_result.semantic_result_fingerprint,
    }
    if benchmark_result.decisions is not None:
        native_reference_comparison = compare_robust_fcr_reference(
            reference_fixture,
            benchmark_result,
        )
        native_reference_comparison_table = pd.DataFrame(
            [field.to_dict() for field in native_reference_comparison.field_results]
        )
        native_reference_benchmark["comparison_status"] = (
            native_reference_comparison.overall_status
        )
        native_reference_benchmark["comparison_fingerprint"] = (
            native_reference_comparison.comparison_fingerprint
        )

        native_source_energy = [
            -value
            for value in benchmark_result.decisions.energy_baseline_charge_positive_kw
        ]
        native_source_fcr = list(benchmark_result.decisions.fcr_capacity_kw)
        source_energy_values = source_energy["displayed_value"].tolist()
        source_fcr_values = source_fcr["displayed_value"].tolist()
        energy_residuals = [
            native - source
            for native, source in zip(native_source_energy, source_energy_values, strict=True)
        ]
        fcr_residuals = [
            native - source
            for native, source in zip(native_source_fcr, source_fcr_values, strict=True)
        ]

        comparison_figure, comparison_axes = plt.subplots(
            2,
            2,
            figsize=(12, 8),
            constrained_layout=True,
        )
        comparison_axes[0, 0].step(hour_indices, source_energy_values, where="mid", label="Julia displayed x0")
        comparison_axes[0, 0].step(hour_indices, native_source_energy, where="mid", linestyle="--", label="native source-frame x0")
        comparison_axes[0, 0].set_ylabel("Energy decision (kW, discharge-positive)")
        comparison_axes[0, 0].set_xlabel("Delivery hour")
        comparison_axes[0, 0].set_title("Hourly decisions")
        comparison_axes[0, 0].legend()

        comparison_axes[0, 1].bar(hour_indices, energy_residuals)
        comparison_axes[0, 1].axhspan(-0.0005, 0.0005, color="tab:green", alpha=0.2, label="source-precision band")
        comparison_axes[0, 1].set_ylabel("Native minus Julia displayed (kW)")
        comparison_axes[0, 1].set_xlabel("Delivery hour")
        comparison_axes[0, 1].set_title("Hourly numerical residual")
        comparison_axes[0, 1].legend()

        comparison_axes[1, 0].step(fcr_block_indices, source_fcr_values, where="mid", label="Julia displayed xr")
        comparison_axes[1, 0].step(fcr_block_indices, native_source_fcr, where="mid", linestyle="--", label="native xr")
        comparison_axes[1, 0].set_ylabel("Symmetric FCR decision (kW)")
        comparison_axes[1, 0].set_xlabel("Four-hour block")
        comparison_axes[1, 0].set_title("FCR decisions")
        comparison_axes[1, 0].legend()

        comparison_axes[1, 1].bar(fcr_block_indices, fcr_residuals)
        comparison_axes[1, 1].axhspan(-0.0005, 0.0005, color="tab:green", alpha=0.2, label="source-precision band")
        comparison_axes[1, 1].set_ylabel("Native minus Julia displayed (kW)")
        comparison_axes[1, 1].set_xlabel("Four-hour block")
        comparison_axes[1, 1].set_title("FCR numerical residual")
        comparison_axes[1, 1].legend()

        generated_figures["native_reference_decision_comparison"] = comparison_figure

display(native_reference_benchmark, native_reference_comparison_table)


## Optional licensed Gurobi reference benchmark

Set RUN_OPTIONAL_GUROBI_REFERENCE_BENCHMARK to True only after installing the optional Gurobi extra and configuring a valid license. This path solves the **same full 2023-06-18 reference restriction** as the optional HiGHS benchmark above, with the same one-hour limit, one thread, seed, and numerical tolerances. It therefore provides an apples-to-apples backend comparison and a more direct reproduction attempt against the recorded Julia/Gurobi result. The resulting figure places the recorded, OEL/Gurobi, and—when available—OEL/HiGHS decisions together, with their residuals from the recorded values alongside them.

This section deliberately solves the restriction rather than the exact bilinear formulation: the governed paper fixture records the restriction, so changing the formulation would confound the solver comparison. OEL still supports explicitly selected Gurobi exact solves as validation evidence through the library API, but the toy exact solve is omitted here because it does not help explain reference-case parity or runtime.

The Gurobi result is optional evidence only. If Gurobi or its license is unavailable, the result remains explicitly unavailable and no fallback to HiGHS occurs.


In [ ]:
gurobi_reference_benchmark = {
    "status": "not_requested",
    "reason": "RUN_OPTIONAL_GUROBI_REFERENCE_BENCHMARK is False",
}
gurobi_reference_comparison_table = pd.DataFrame()
gurobi_reference_backend_table = pd.DataFrame()
if RUN_OPTIONAL_GUROBI_REFERENCE_BENCHMARK:
    gurobi_policy = RobustFCRSolvePolicy(
        solver_name="gurobi",
        time_limit_seconds=GUROBI_REFERENCE_BENCHMARK_TIME_LIMIT_SECONDS,
        threads=1,
        random_seed=13,
        primal_feasibility_tolerance=1e-7,
        dual_feasibility_tolerance=1e-7,
        mip_feasibility_tolerance=1e-6,
        relative_mip_gap=1e-6,
        objective_comparison_absolute_tolerance_eur=1e-6,
        objective_comparison_relative_tolerance=1e-9,
        allow_feasible_incumbent_plan=False,
    )
    gurobi_reference_problem = build_robust_fcr_reference_problem(
        reference_fixture,
        solve_policy=gurobi_policy,
    )
    gurobi_reference_result = solve_robust_fcr_bid(
        gurobi_reference_problem,
        formulation="restriction",
    )
    gurobi_reference_benchmark = {
        "status": gurobi_reference_result.solver.solve_status,
        "termination": gurobi_reference_result.solver.termination_status,
        "objective_eur": gurobi_reference_result.solver.objective_value_eur,
        "best_upper_bound_eur": gurobi_reference_result.solver.best_upper_bound_eur,
        "relative_gap": gurobi_reference_result.solver.relative_gap,
        "runtime_seconds": gurobi_reference_result.solver.runtime_seconds,
        "deployability": gurobi_reference_result.deployability_status,
        "result_fingerprint": gurobi_reference_result.semantic_result_fingerprint,
    }
    if gurobi_reference_result.decisions is not None:
        gurobi_reference_comparison = compare_robust_fcr_reference(
            reference_fixture,
            gurobi_reference_result,
        )
        gurobi_reference_comparison_table = pd.DataFrame(
            [field.to_dict() for field in gurobi_reference_comparison.field_results]
        )
        gurobi_reference_benchmark["comparison_status"] = (
            gurobi_reference_comparison.overall_status
        )
        gurobi_reference_benchmark["comparison_fingerprint"] = (
            gurobi_reference_comparison.comparison_fingerprint
        )

    backend_rows = [
        {
            "backend": "recorded Julia/Gurobi",
            "status": "recorded",
            "objective (EUR)": reference_fixture.recorded_result.objective_value_eur,
            "best upper bound (EUR)": None,
            "relative gap": None,
            "runtime (s)": reference_fixture.recorded_result.solve_time_seconds,
        },
    ]
    if benchmark_result is not None:
        backend_rows.append(
            {
                "backend": "OEL / HiGHS",
                "status": benchmark_result.solver.solve_status,
                "objective (EUR)": benchmark_result.solver.objective_value_eur,
                "best upper bound (EUR)": benchmark_result.solver.best_upper_bound_eur,
                "relative gap": benchmark_result.solver.relative_gap,
                "runtime (s)": benchmark_result.solver.runtime_seconds,
            }
        )
    backend_rows.append(
        {
            "backend": "OEL / Gurobi",
            "status": gurobi_reference_result.solver.solve_status,
            "objective (EUR)": gurobi_reference_result.solver.objective_value_eur,
            "best upper bound (EUR)": gurobi_reference_result.solver.best_upper_bound_eur,
            "relative gap": gurobi_reference_result.solver.relative_gap,
            "runtime (s)": gurobi_reference_result.solver.runtime_seconds,
        }
    )
    gurobi_reference_backend_table = pd.DataFrame(backend_rows)

    if gurobi_reference_result.decisions is not None:
        recorded_energy = source_energy["displayed_value"].tolist()
        recorded_fcr = source_fcr["displayed_value"].tolist()
        gurobi_energy = [
            -value
            for value in gurobi_reference_result.decisions.energy_baseline_charge_positive_kw
        ]
        gurobi_fcr = list(gurobi_reference_result.decisions.fcr_capacity_kw)
        gurobi_energy_residual = [
            native - recorded
            for native, recorded in zip(gurobi_energy, recorded_energy, strict=True)
        ]
        gurobi_fcr_residual = [
            native - recorded
            for native, recorded in zip(gurobi_fcr, recorded_fcr, strict=True)
        ]

        gurobi_reference_figure, gurobi_reference_axes = plt.subplots(
            2,
            2,
            figsize=(13, 8),
            constrained_layout=True,
        )
        gurobi_reference_axes[0, 0].step(
            hour_indices, recorded_energy, where="mid", label="recorded Julia/Gurobi x0"
        )
        gurobi_reference_axes[0, 0].step(
            hour_indices, gurobi_energy, where="mid", linestyle="--", label="OEL/Gurobi x0"
        )
        gurobi_reference_axes[0, 1].bar(
            [index - 0.2 for index in hour_indices],
            gurobi_energy_residual,
            width=0.4,
            label="OEL/Gurobi minus recorded",
        )
        gurobi_reference_axes[1, 0].step(
            fcr_block_indices, recorded_fcr, where="mid", label="recorded Julia/Gurobi xr"
        )
        gurobi_reference_axes[1, 0].step(
            fcr_block_indices, gurobi_fcr, where="mid", linestyle="--", label="OEL/Gurobi xr"
        )
        gurobi_reference_axes[1, 1].bar(
            [index - 0.2 for index in fcr_block_indices],
            gurobi_fcr_residual,
            width=0.4,
            label="OEL/Gurobi minus recorded",
        )

        if benchmark_result is not None and benchmark_result.decisions is not None:
            highs_energy = [
                -value
                for value in benchmark_result.decisions.energy_baseline_charge_positive_kw
            ]
            highs_fcr = list(benchmark_result.decisions.fcr_capacity_kw)
            gurobi_reference_axes[0, 0].step(
                hour_indices, highs_energy, where="mid", linestyle=":", label="OEL/HiGHS x0"
            )
            gurobi_reference_axes[0, 1].bar(
                [index + 0.2 for index in hour_indices],
                [native - recorded for native, recorded in zip(highs_energy, recorded_energy, strict=True)],
                width=0.4,
                label="OEL/HiGHS minus recorded",
            )
            gurobi_reference_axes[1, 0].step(
                fcr_block_indices, highs_fcr, where="mid", linestyle=":", label="OEL/HiGHS xr"
            )
            gurobi_reference_axes[1, 1].bar(
                [index + 0.2 for index in fcr_block_indices],
                [native - recorded for native, recorded in zip(highs_fcr, recorded_fcr, strict=True)],
                width=0.4,
                label="OEL/HiGHS minus recorded",
            )

        for axis in (gurobi_reference_axes[0, 1], gurobi_reference_axes[1, 1]):
            axis.axhspan(
                -0.0005, 0.0005, color="tab:green", alpha=0.2, label="source-precision band"
            )
        gurobi_reference_axes[0, 0].set_title("Hourly energy decisions")
        gurobi_reference_axes[0, 1].set_title("Hourly residuals from recorded decisions")
        gurobi_reference_axes[1, 0].set_title("Four-hour FCR decisions")
        gurobi_reference_axes[1, 1].set_title("FCR residuals from recorded decisions")
        gurobi_reference_axes[0, 0].set_ylabel("Energy decision (kW, discharge-positive)")
        gurobi_reference_axes[0, 1].set_ylabel("Native minus recorded (kW)")
        gurobi_reference_axes[1, 0].set_ylabel("Symmetric FCR decision (kW)")
        gurobi_reference_axes[1, 1].set_ylabel("Native minus recorded (kW)")
        for axis in gurobi_reference_axes[0, :]:
            axis.set_xlabel("Delivery hour")
        for axis in gurobi_reference_axes[1, :]:
            axis.set_xlabel("Four-hour block")
        for axis in gurobi_reference_axes.flat:
            axis.legend(loc="best")

        generated_figures["gurobi_reference_backend_comparison"] = gurobi_reference_figure
        display(gurobi_reference_figure)
        plt.close(gurobi_reference_figure)

display(
    gurobi_reference_benchmark,
    gurobi_reference_backend_table,
    gurobi_reference_comparison_table,
)


## Optional explicit-path historical activation replay

Set RUN_EXTERNAL_ACTIVATION_REPLAY to True only when the governed external CSV is installed at the configured path. This section replays the recorded fixed reference plan through Feature 012; it never exposes activation to Feature 013 and never searches for, downloads, infers, or substitutes the file.

The selected external-reference workflow remains the place for complete historical physical comparison. The authored stress replays above demonstrate the native contract boundary, not historical trace parity.


In [ ]:
external_activation_result = {
    "status": "not_requested",
    "reason": "RUN_EXTERNAL_ACTIVATION_REPLAY is False",
}
if RUN_EXTERNAL_ACTIVATION_REPLAY:
    if not EXTERNAL_ACTIVATION_PATH.is_file():
        raise FileNotFoundError(
            "explicit reference activation CSV is absent at "
            f"{EXTERNAL_ACTIVATION_PATH}; install the governed file or disable "
            "RUN_EXTERNAL_ACTIVATION_REPLAY"
        )
    from oel_validation.scripts.run_fcr_bid_reference_parity import (
        build_reference_request,
    )
    external_request = build_reference_request(
        fixture_path=EVALUATION_FIXTURE_PATH,
        activation_path=EXTERNAL_ACTIVATION_PATH,
    )
    external_report = evaluate_fixed_fcr_bid(external_request)
    external_activation_result = {
        "status": external_report.status,
        "activation_count": len(external_request.activation_trace.samples),
        "trace_fingerprint": external_report.trace_fingerprint,
        "settlement_fingerprint": external_report.settlement_fingerprint,
        "result_fingerprint": external_report.result_fingerprint,
        "total_profit_eur": external_report.total_profit_eur,
    }

external_activation_result


In [ ]:
exported_figure_paths = []
if EXPORT_FIGURES:
    FIGURE_EXPORT_DIR.mkdir(parents=True, exist_ok=True)
    for figure_name, figure in generated_figures.items():
        destination = FIGURE_EXPORT_DIR / f"{figure_name}.png"
        figure.savefig(destination, dpi=160, bbox_inches="tight")
        exported_figure_paths.append(destination)

exported_figure_paths


In [ ]:
notebook_display_payload = {
    "problem_fingerprint": walkthrough_problem.problem_fingerprint,
    "restriction_result_fingerprint": restriction_result.semantic_result_fingerprint,
    "relaxation_result_fingerprint": relaxation_result.semantic_result_fingerprint,
    "bound_comparison_fingerprint": bound_comparison.comparison_fingerprint,
    "decision_vector_fingerprint": restriction_result.decisions.decision_vector_fingerprint,
    "plan_fingerprint": restriction_result.deployable_plan.plan_fingerprint,
    "activation_trace_fingerprints": {
        case: walkthrough_evaluation_requests[case].activation_trace.artifact_fingerprint
        for case in activation_cases
    },
    "physical_trace_fingerprints": {
        case: walkthrough_evaluation_reports[case].trace_fingerprint
        for case in activation_cases
    },
    "settlement_fingerprints": {
        case: walkthrough_evaluation_reports[case].settlement_fingerprint
        for case in activation_cases
    },
    "evaluation_result_fingerprints": {
        case: walkthrough_evaluation_reports[case].result_fingerprint
        for case in activation_cases
    },
    "reference_fixture_fingerprint": reference_fixture.fixture_fingerprint,
    "reference_decision_content_fingerprint": (
        reference_fixture.recorded_result.decision_content_fingerprint
    ),
    "generated_figure_names": sorted(generated_figures),
}
notebook_display_fingerprint = stable_fingerprint(notebook_display_payload)
display_summary_table = pd.DataFrame(
    [
        ("notebook display fingerprint", notebook_display_fingerprint),
        ("default figure count", len(generated_figures)),
        ("settled Feature 012 scenarios", len(walkthrough_evaluation_reports)),
        ("native reference benchmark", native_reference_benchmark["status"]),
        ("optional Gurobi reference benchmark", gurobi_reference_benchmark["status"]),
        ("external activation replay", external_activation_result["status"]),
    ],
    columns=["evidence", "value"],
)
display_summary_table


## Conclusions

- The example now shows a real joint decision: arbitrage fills and empties the battery at selected hours, leaving no reserve in the first two blocks and only 3.636 kW later.
- The constraint-attribution plot distinguishes converter headroom from robust stored-energy protection. A 10 kW converter does not imply that 10 kW of symmetric reserve is deliverable for the declared uncertainty budget.
- The restriction alone emits an audited robust-labelled FixedFCRBidPlan. The relaxation remains upper-bound evidence and emits no deployable plan, even though both formulations happen to meet at the same optimum in this hand-calculable case.
- Feature 012 replays the unchanged plan under nominal, unseen upward-pulse, and unseen downward-pulse activation. All three settle without tracking error or saturation, while stored energy and realized profit change with activation.
- The normalized Julia/Gurobi fixture supports provenance-rich, source-precision-bounded comparison only when the identical historical problem is solved explicitly. It is not compared to this teaching case.

Rolling or multi-day studies, SOC carry-over, intraday recourse, market-acceptance proof, TPM fitting or bid selection, degradation costs, aFRR/mFRR, live data acquisition, and full-history paper experiments remain out of scope. Exact unrounded schedule equality, source optimality proof, and complete Julia trace parity remain unavailable from the normalized fixture alone.
